In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt 
import seaborn as sns
import os
import tensorflow as tf
from tensorflow import keras
import torch
import torchvision
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torchvision.transforms import transforms,ToTensor,Lambda,Compose
from torchvision import datasets
import torch.optim as optim

In [41]:
# Download training data from open datasets
train_data = datasets.FashionMNIST(
    root="data",
    train=True,
    download=True,
    transform=ToTensor(),
)

# Download test data from open datasets
test_data = datasets.FashionMNIST(
    root="data",
    train=False,
    download=True,
    transform=ToTensor(),
)

In [42]:
custom_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


In [43]:
from PIL import Image

class CustomDataset(Dataset):
    def __init__(self,features,labels,transform):
        self.features=features
        self.labels=labels
        self.transform=transform
    
    def __len__(self):
        return len(self.features)
    
    def __getitem__(self,idx):

        # resize image
        image=self.features[idx].reshape(28,28)
        # change datatype as unit8
        image=image.astype(np.uint8)
        # change black and white to RGB (H,W,C)-->(C,H,W)
        image =np.stack([image]*3,axis=-1)
        #convert array to PIL image
        image=Image.fromarray(image)
        # apply transformation
        image=self.transform(image)

        return image,torch.tensor(self.labels[idx],dtype=torch.long)

In [44]:
train_dataset=CustomDataset(train_data.data.numpy(),train_data.targets.numpy(),
                            transform=custom_transform)
test_dataset=CustomDataset(test_data.data.numpy(),test_data.targets.numpy(),
                           transform=custom_transform)

In [45]:
batch_size=64
train_dataloader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
test_dataloader  = DataLoader(test_data, batch_size=batch_size, shuffle=False)


In [46]:
import torchvision.models as models
vgg16 = models.vgg16(pretrained=True)



In [47]:
vgg16.classifier

Sequential(
  (0): Linear(in_features=25088, out_features=4096, bias=True)
  (1): ReLU(inplace=True)
  (2): Dropout(p=0.5, inplace=False)
  (3): Linear(in_features=4096, out_features=4096, bias=True)
  (4): ReLU(inplace=True)
  (5): Dropout(p=0.5, inplace=False)
  (6): Linear(in_features=4096, out_features=1000, bias=True)
)

In [49]:
for parm in vgg16.parameters():
    parm.requires_grad = False



In [50]:
vgg16.classifier=nn.Sequential(
    nn.Linear(25088,1024),
    nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(1024,512),
    nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(512,10)
)

In [51]:
learning_rate=0.001
epochs=10

In [52]:
images, labels = next(iter(train_dataloader))
print(images.shape)


torch.Size([64, 1, 28, 28])


In [53]:
criterion=nn.CrossEntropyLoss()
optimizer=optim.Adam(vgg16.classifier.parameters(),lr=learning_rate)

In [54]:
for epochs in range(epochs):
    total_epochs_loss=0
    for batch_features,batch_labels in train_dataloader:
        # zero the parameter gradients
        optimizer.zero_grad()

        # forward pass
        outputs=vgg16(batch_features)
        loss=criterion(outputs,batch_labels)

        # backward pass and optimization
        loss.backward()
        optimizer.step()

        total_epochs_loss+=loss.item()
    
    avg_loss=total_epochs_loss/len(train_dataloader)
    print(f"Epoch [{epochs+1}], Loss: {avg_loss:.4f}")

RuntimeError: Given groups=1, weight of size [64, 3, 3, 3], expected input[64, 1, 28, 28] to have 3 channels, but got 1 channels instead

In [55]:
model.eval()

NameError: name 'model' is not defined

In [56]:
#evalution on test data
total=0
correct=0

with torch.no_grad():
    for batch_features,batch_labels in test_dataloader:
        outputs=vgg16(batch_features)
        _,predicted=torch.max(outputs.data,1)
        total+=batch_labels.size[0]
        correct+=(predicted==batch_labels).sum().item()
    print(f"Accuracy on test data: {100*correct/total}%")

RuntimeError: Given groups=1, weight of size [64, 3, 3, 3], expected input[64, 1, 28, 28] to have 3 channels, but got 1 channels instead